In [ ]:
import os
import torch
from sparse_autoencoder import SparseAutoencoder

def _infer_dims_from_state_dict(sd: dict):
    # Try to infer (n_input_features, n_learned_features) from linear weight shapes.
    # Works for typical encoder/decoder pairs: encoder W ~ [n_learned, n_input], decoder W ~ [n_input, n_learned]
    enc_shape = dec_shape = None
    for k, v in sd.items():
        if not (k.endswith(".weight") and v.ndim == 2):
            continue
        # Heuristic: keep the smallest few by name
        if "enc" in k or "encoder" in k:
            enc_shape = tuple(v.shape)
        if "dec" in k or "decoder" in k:
            dec_shape = tuple(v.shape)
    if enc_shape and dec_shape:
        # Prefer mutual consistency
        if enc_shape[1] == dec_shape[0] and enc_shape[0] == dec_shape[1]:
            n_input, n_learned = dec_shape[0], dec_shape[1]
            return n_input, n_learned
    # Fallback: best guess from any two 2D weights
    weights = [(k, v.shape) for k, v in sd.items() if k.endswith(".weight") and sd[k].ndim == 2]
    if len(weights) >= 1:
        k0, s0 = weights[0]
        # assume decoder-like [n_input, n_learned]
        n_input, n_learned = s0[0], s0[1]
        return n_input, n_learned
    raise ValueError("Could not infer dims from state_dict.")

def load_sae_and_print_keys(checkpoint_path: str, device: str = None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    obj = torch.load(checkpoint_path, map_location=device, weights_only=True)

    # Case A: full model was saved
    if isinstance(obj, SparseAutoencoder):
        model = obj.to(device)
        state_dict = model.state_dict()
        print(f"Loaded full model from: {checkpoint_path}")
        print(f"Model type: {type(model).__name__}")
        print("== state_dict keys ==")
        for k in state_dict.keys():
            print(k)
        return model

    # Case B: checkpoint dict
    if isinstance(obj, dict):
        # Common key names
        for key in ("autoencoder_state_dict", "model_state_dict", "state_dict", "autoencoder"):
            if key in obj and isinstance(obj[key], dict):
                state_dict = obj[key]
                break
        else:
            # Maybe the dict *is* a raw state_dict
            if all(isinstance(v, torch.Tensor) for v in obj.values()):
                state_dict = obj
            else:
                raise ValueError(f"Unrecognized checkpoint structure: keys={list(obj.keys())[:10]}")

        # Try to get dims from checkpoint hparams first
        n_input = obj.get("n_input_features")
        n_learned = obj.get("n_learned_features")
        if n_input is None or n_learned is None:
            n_input, n_learned = _infer_dims_from_state_dict(state_dict)

        n_components = obj.get("n_components", 1)

        model = SparseAutoencoder(
            n_input_features=int(n_input),
            n_learned_features=int(n_learned),
            n_components=int(n_components),
        ).to(device)

        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print(f"Loaded state_dict from: {checkpoint_path}")
        print(f"inferred n_input_features={n_input}, n_learned_features={n_learned}, n_components={n_components}")
        if missing:
            print(f"[warn] Missing keys ({len(missing)}): {missing[:10]}{' ...' if len(missing)>10 else ''}")
        if unexpected:
            print(f"[warn] Unexpected keys ({len(unexpected)}): {unexpected[:10]}{' ...' if len(unexpected)>10 else ''}")

        print("== state_dict keys ==")
        for k in state_dict.keys():
            print(k)
        return model

    raise TypeError(f"Unsupported checkpoint type: {type(obj)}")

# ---- usage ----
# Point this to a file under: f"{args.save_dir_sae_ckpts[args.modality]}{args.save_suffix}/..."

_ = load_sae_and_print_keys(sae_ckpt_path)


/home/sunayana/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sunayana/miniconda3/envs/dncbm310/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


ValueError: Could not infer dims from state_dict.

In [3]:
import torch
from sparse_autoencoder import SparseAutoencoder

def _extract_state_dict(obj):
    if isinstance(obj, SparseAutoencoder):
        return obj.state_dict(), obj
    if isinstance(obj, dict):
        for k in ("state_dict", "model_state_dict", "autoencoder_state_dict", "autoencoder"):
            if k in obj and isinstance(obj[k], dict):
                return obj[k], None
        if all(isinstance(v, torch.Tensor) for v in obj.values()):
            return obj, None
    raise TypeError(f"Unsupported checkpoint: type={type(obj)}, keys={list(obj.keys())[:10] if isinstance(obj, dict) else 'N/A'}")

def _infer_dims_with_components(sd: dict):
    enc3 = dec3 = bias2 = None
    for k, v in sd.items():
        if not isinstance(v, torch.Tensor):
            continue
        if v.ndim == 3:
            # Encoder: [C, F, D]; Decoder: [C, D, F]
            if "enc" in k or "encoder" in k:
                enc3 = v.shape  # (C,F,D)
            if "dec" in k or "decoder" in k:
                dec3 = v.shape  # (C,D,F)
        elif v.ndim == 2:
            if "bias" in k or "tied_bias" in k:
                bias2 = v.shape  # (C,D) usually

    if enc3 and dec3:
        C, F, D = enc3
        # sanity check with decoder
        assert dec3[0] == C and dec3[1] == D and dec3[2] == F, f"Encoder/decoder shapes disagree: enc={enc3}, dec={dec3}"
        return D, F, C
    if enc3:  # only encoder seen
        C, F, D = enc3
        return D, F, C
    if dec3:  # only decoder seen
        C, D, F = dec3
        return D, F, C
    if bias2:  # fallback: bias has [C, D]
        C, D = bias2
        # Need F; try to guess from any 2D weight [?, F] or [F, ?]
        F = None
        for k, v in sd.items():
            if isinstance(v, torch.Tensor) and v.ndim == 2 and v.shape[0] != C and v.shape[1] != D:
                # pick the other dimension as F candidate
                if v.shape[0] == D: F = v.shape[1]
                elif v.shape[1] == D: F = v.shape[0]
        if F is None:
            raise ValueError("Could not infer n_learned_features from 2D tensors.")
        return D, F, C

    # Last resort: look for largest 3D tensor
    mats3 = [(k, v.shape) for k, v in sd.items() if isinstance(v, torch.Tensor) and v.ndim == 3]
    if mats3:
        k, (C, A, B) = max(mats3, key=lambda kv: kv[1][0]*kv[1][1]*kv[1][2])
        # Assume encoder layout if name suggests, else decoder
        if "enc" in k or "encoder" in k:  # [C,F,D]
            return B, A, C
        else:  # assume decoder [C,D,F]
            return A, B, C

    raise ValueError("Could not infer dims: no suitable 3D/2D tensors found.")

def load_sae_and_print_keys(checkpoint_path, device=None, explicit_dims=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
    sd, model = _extract_state_dict(ckpt)

    print(f"Found {len(sd)} state_dict entries. Sample (up to 25):")
    for i, (k, v) in enumerate(sd.items()):
        if i >= 25: break
        shape = tuple(v.shape) if isinstance(v, torch.Tensor) else "non-tensor"
        print(f"  {k}: {shape}")

    if model is not None:
        print("Loaded full SparseAutoencoder model; listing keys:")
        for k in model.state_dict().keys():
            print(" ", k)
        return model.to(device)

    if explicit_dims is None:
        n_in, n_feat, n_comp = _infer_dims_with_components(sd)
    else:
        n_in, n_feat = explicit_dims
        # Try to infer n_comp from any 3D tensor; default 1 if absent
        any3 = [v for v in sd.values() if isinstance(v, torch.Tensor) and v.ndim == 3]
        n_comp = any3[0].shape[0] if any3 else 1

    print(f"Using dims → n_input_features={n_in}, n_learned_features={n_feat}, n_components={n_comp}")

    model = SparseAutoencoder(
        n_input_features=int(n_in),
        n_learned_features=int(n_feat),
        n_components=int(n_comp),
    ).to(device)

    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:   print(f"[warn] Missing keys ({len(missing)}): {missing[:12]}{' ...' if len(missing)>12 else ''}")
    if unexpected: print(f"[warn] Unexpected keys ({len(unexpected)}): {unexpected[:12]}{' ...' if len(unexpected)>12 else ''}")

    print("== All loaded keys ==")
    for k in sd.keys():
        print(k)
    return model

# --- usage for your file ---
ckpt = "/home/sunayana/Documents/Concept_LoRA/Discover-then-Name/pretrained/Checkpoints/clip_ViT-B:16_sparse_autoencoder_final.pt"
model = load_sae_and_print_keys(ckpt)


Found 6 state_dict entries. Sample (up to 25):
  tied_bias: (1, 512)
  pre_encoder_bias._bias_reference: (1, 512)
  encoder._weight: (1, 4096, 512)
  encoder._bias: (1, 4096)
  decoder._weight: (1, 512, 4096)
  post_decoder_bias._bias_reference: (1, 512)
Using dims → n_input_features=512, n_learned_features=4096, n_components=1
== All loaded keys ==
tied_bias
pre_encoder_bias._bias_reference
encoder._weight
encoder._bias
decoder._weight
post_decoder_bias._bias_reference


In [4]:
import torch

def describe_sae_from_state(sd):
    def shp(k): return tuple(sd[k].shape) if k in sd else None
    keys = list(sd.keys())
    print(f"Found {len(keys)} tensors")

    info = {
        "tied_bias": shp("tied_bias"),
        "pre_bias_ref": shp("pre_encoder_bias._bias_reference"),
        "enc_W": shp("encoder._weight"),
        "enc_b": shp("encoder._bias"),
        "dec_W": shp("decoder._weight"),
        "post_bias_ref": shp("post_decoder_bias._bias_reference"),
    }
    for k,v in info.items(): print(f"{k:18s}: {v}")

    # Infer dims
    C = info["enc_W"][0] if info["enc_W"] is not None else (info["dec_W"][0] if info["dec_W"] else info["tied_bias"][0] if info["tied_bias"] and len(info["tied_bias"])==2 else 1)
    F = info["enc_W"][1] if info["enc_W"] else (info["dec_W"][2] if info["dec_W"] else None)
    D = info["enc_W"][2] if info["enc_W"] else (info["dec_W"][1] if info["dec_W"] else (info["tied_bias"][1] if info["tied_bias"] else None))
    print(f"\nInferred -> n_components={C}, n_input_features={D}, n_learned_features={F}")

    # Sanity checks
    if info["enc_W"] and info["dec_W"]:
        C1,F1,D1 = info["enc_W"]
        C2,D2,F2 = info["dec_W"]
        assert (C1,C2)==(C,C) and (F1,F2)==(F,F) and (D1,D2)==(D,D), "Encoder/decoder shapes disagree!"
        print("Shapes consistent: [C,F,D] encoder, [C,D,F] decoder")

    # Unit-norm check (decoder columns)
    if "decoder._weight" in sd:
        W_dec = sd["decoder._weight"].squeeze(0)  # [D,F] if C=1
        col_norms = W_dec.norm(dim=0)
        print(f"Decoder column norms: mean={col_norms.mean().item():.4f}  min={col_norms.min().item():.4f}  max={col_norms.max().item():.4f}")

def forward_equations():
    print(
"""Forward (per component):
  x̃ = x + b_tied                   # pre_encoder_bias
  z  = x̃ @ W_encᵀ + b_enc         # encoder (Linear)
  x̂  = z  @ W_decᵀ + b_tied       # decoder + post_decoder_bias
Constraints:
  UnitNormDecoder ⇒ columns of W_dec are ||·||₂ = 1 (enforced post-backward).
"""
)

# Usage:
sd = model.state_dict()  # or torch.load(...) if you stored raw sd
describe_sae_from_state(sd)
forward_equations()

Found 6 tensors
tied_bias         : (1, 512)
pre_bias_ref      : (1, 512)
enc_W             : (1, 4096, 512)
enc_b             : (1, 4096)
dec_W             : (1, 512, 4096)
post_bias_ref     : (1, 512)

Inferred -> n_components=1, n_input_features=512, n_learned_features=4096
Shapes consistent: [C,F,D] encoder, [C,D,F] decoder
Decoder column norms: mean=1.0000  min=1.0000  max=1.0000
Forward (per component):
  x̃ = x + b_tied                   # pre_encoder_bias
  z  = x̃ @ W_encᵀ + b_enc         # encoder (Linear)
  x̂  = z  @ W_decᵀ + b_tied       # decoder + post_decoder_bias
Constraints:
  UnitNormDecoder ⇒ columns of W_dec are ||·||₂ = 1 (enforced post-backward).



In [6]:
import torch
from sparse_autoencoder import SparseAutoencoder

def build_model_from_sd(sd):
    # infer dims (handles your shapes)
    if "encoder._weight" in sd and sd["encoder._weight"].ndim == 3:
        C, F, D = sd["encoder._weight"].shape          # [C, F, D]
    elif "decoder._weight" in sd and sd["decoder._weight"].ndim == 3:
        C, D, F = sd["decoder._weight"].shape          # [C, D, F]
    else:
        C, D = sd["tied_bias"].shape                   # [C, D]
        F = sd["encoder._bias"].shape[-1] if "encoder._bias" in sd else sd["decoder._weight"].shape[-1]

    # IMPORTANT: keep the axis even if C==1
    model = SparseAutoencoder(
        n_input_features=int(D),
        n_learned_features=int(F),
        n_components=int(C),        # <- do NOT switch to None when C==1
    )
    model.load_state_dict(sd, strict=False)
    return model

def print_layerwise(model):
    print("== Module tree ==")
    for n, m in model.named_modules():
        print(n or "(root)", "::", m.__class__.__name__)
    print("\n== Parameters ==")
    for n, p in model.named_parameters():
        print(f"{n:40s} {list(p.shape)}")
    print("\n== Buffers ==")
    for n, b in model.named_buffers():
        print(f"{n:40s} {list(b.shape)}")

# --- usage ---
ckpt = "/home/sunayana/Documents/Concept_LoRA/Discover-then-Name/pretrained/Checkpoints/clip_ViT-B:16_sparse_autoencoder_final.pt"
obj = torch.load(ckpt, map_location="cpu", weights_only=True)
sd  = obj["state_dict"] if isinstance(obj, dict) and "state_dict" in obj else (obj if isinstance(obj, dict) else obj.state_dict())

model = build_model_from_sd(sd)     # now loads without size-mismatch
print_layerwise(model)


== Module tree ==
(root) :: SparseAutoencoder
pre_encoder_bias :: TiedBias
encoder :: LinearEncoder
encoder.activation_function :: ReLU
decoder :: UnitNormDecoder
post_decoder_bias :: TiedBias

== Parameters ==
tied_bias                                [1, 512]
encoder._weight                          [1, 4096, 512]
encoder._bias                            [1, 4096]
decoder._weight                          [1, 512, 4096]

== Buffers ==


In [ ]:
import torch
from sparse_autoencoder import SparseAutoencoder

def build_model_from_sd(sd):
    # infer dims (handles your shapes)
    if "encoder._weight" in sd and sd["encoder._weight"].ndim == 3:
        C, F, D = sd["encoder._weight"].shape          # [C, F, D]
    elif "decoder._weight" in sd and sd["decoder._weight"].ndim == 3:
        C, D, F = sd["decoder._weight"].shape          # [C, D, F]
    else:
        C, D = sd["tied_bias"].shape                   # [C, D]
        F = sd["encoder._bias"].shape[-1] if "encoder._bias" in sd else sd["decoder._weight"].shape[-1]

    # IMPORTANT: keep the axis even if C==1
    model = SparseAutoencoder(
        n_input_features=int(D),
        n_learned_features=int(F),
        n_components=int(C),        # <- do NOT switch to None when C==1
    )
    model.load_state_dict(sd, strict=False)
    return model

def print_layerwise(model):
    print("== Module tree ==")
    for n, m in model.named_modules():
        print(n or "(root)", "::", m.__class__.__name__)
    print("\n== Parameters ==")
    for n, p in model.named_parameters():
        print(f"{n:40s} {list(p.shape)}")
    print("\n== Buffers ==")
    for n, b in model.named_buffers():
        print(f"{n:40s} {list(b.shape)}")

# --- usage ---
sae_ckpt_path_pets="/home/sunayana/Documents/Concept_LoRA/Discover-then-Name/SAE/SAEImg/cc3m/clip_RN50/out/lr0.0005_l1coeff3e-05_ef8_rf10_hookout_bs4096_epo200/sae_checkpoints/sparse_autoencoder_final.pt"
obj = torch.load(ckpt, map_location="cpu", weights_only=True)
sd  = obj["state_dict"] if isinstance(obj, dict) and "state_dict" in obj else (obj if isinstance(obj, dict) else obj.state_dict())

model = build_model_from_sd(sd)     # now loads without size-mismatch
print_layerwise(model)




== Module tree ==
(root) :: SparseAutoencoder
pre_encoder_bias :: TiedBias
encoder :: LinearEncoder
encoder.activation_function :: ReLU
decoder :: UnitNormDecoder
post_decoder_bias :: TiedBias

== Parameters ==
tied_bias                                [1, 512]
encoder._weight                          [1, 4096, 512]
encoder._bias                            [1, 4096]
decoder._weight                          [1, 512, 4096]

== Buffers ==


In [2]:
import os, torch
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import OxfordIIITPet, ImageFolder
import clip

# --- paths (edit to your absolute paths) ---
PETS_BASE = "/home/sunayana/Documents/Concept_LoRA/datasets"
PETS_OFFICIAL = os.path.join(PETS_BASE, "the-oxfordiiit-pet-dataset")  # must contain images/ and annotations/
PETS_STRUCTURED = os.path.join(PETS_BASE, "pets_structured")            # your class-per-folder layout
SAE_CKPT = "/home/sunayana/Documents/Concept_LoRA/Discover-then-Name/SAE/SAEImg/pets/clip_ViT-B16/out/lr0.0001_l1coeff0.0003_ef4_rf500000_hookout_bs4096_epo200/sae_checkpoints_pets_vitb16_d512/sparse_autoencoder_final.pt"
# --- CLIP encoder ---
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()

# --- dataset loader chooser ---
def make_loader(batch_size=64, workers=4):
    if os.path.isdir(os.path.join(PETS_OFFICIAL, "images")) and os.path.isdir(os.path.join(PETS_OFFICIAL, "annotations")):
        ds = OxfordIIITPet(PETS_OFFICIAL, split="test", target_types="category", download=False, transform=preprocess)
        print("Using OxfordIIITPet (official layout).")
    else:
        # Fallback: your structured layout (class folders)
        ds = ImageFolder(PETS_STRUCTURED, transform=preprocess)
        print("Using ImageFolder on pets_structured/. Classes:", len(ds.classes))
    return ds, DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)

# --- load SAE (keeps component axis) ---
from sparse_autoencoder import SparseAutoencoder
def load_sae_from_ckpt(path, device=device):
    obj = torch.load(path, map_location=device, weights_only=True)
    sd = obj["state_dict"] if isinstance(obj, dict) and "state_dict" in obj else (obj if isinstance(obj, dict) else obj.state_dict())
    if "encoder._weight" in sd and sd["encoder._weight"].ndim == 3:
        C, F, D = sd["encoder._weight"].shape
    elif "decoder._weight" in sd and sd["decoder._weight"].ndim == 3:
        C, D, F = sd["decoder._weight"].shape
    else:
        C, D = sd["tied_bias"].shape; F = sd["encoder._bias"].shape[-1]
    sae = SparseAutoencoder(n_input_features=int(D), n_learned_features=int(F), n_components=int(C)).to(device)
    sae.load_state_dict(sd, strict=False); sae.eval()
    return sae, D, F, C

@torch.no_grad()
def encode_images_to_clip(imgs):
    x = clip_model.encode_image(imgs)
    x = x / x.norm(dim=-1, keepdim=True)
    return x

#load pets dataset from the root using ImageFolder
if os.path.isdir(os.path.join(PETS_OFFICIAL, "images")) and os.path.isdir(os.path.join(PETS_OFFICIAL, "annotations")):
    ds = OxfordIIITPet(PETS_OFFICIAL, split="test", target_types="category", download=False, transform=preprocess)
    print("Using OxfordIIITPet (official layout).")
else:
    # Fallback: your structured layout (class folders)
    ds = ImageFolder(PETS_STRUCTURED, transform=preprocess)
    print("Using ImageFolder on pets_structured/. Classes:", len(ds.classes)) 
      
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)



sae, D, F, C = load_sae_from_ckpt(SAE_CKPT, device)
print(f"SAE dims: D={D}, F={F}, C={C}")

all_acts = []
all_paths = []

with torch.no_grad():
    for imgs, targets in loader:
        imgs = imgs.to(device)
        feats = encode_images_to_clip(imgs)       # [B, D]
        feats = feats.unsqueeze(1)                # [B, 1, D] for C=1
        la, _ = sae(feats)                        # learned activations [B, 1, F]
        all_acts.append(la.squeeze(1).cpu())      # [B, F]
        # robust path extraction
        if hasattr(ds, "imgs"):
            all_paths += [p for p,_ in ds.imgs[len(all_paths):len(all_paths)+imgs.size(0)]]
        elif hasattr(ds, "_images"):
            all_paths += [os.path.join(PETS_OFFICIAL, "images", fn) for fn,_ in ds._images[len(all_paths):len(all_paths)+imgs.size(0)]]
        else:
            all_paths += ["<path NA>"] * imgs.size(0)

acts = torch.cat(all_acts, dim=0)                 # [N, F]

# ----- top concepts (dataset-wide mean activation) -----
topk = 20
vals, idxs = torch.topk(acts.mean(dim=0), k=topk)
print("\nTop concepts by mean activation:")
for v, i in zip(vals.tolist(), idxs.tolist()):
    print(f"  concept_{i:04d}: mean_act={v:.4f}")

# ----- top images for the strongest concept -----
K = 12
j = idxs[0].item()
col = acts[:, j]
v2, i2 = torch.topk(col, k=min(K, col.numel()))
print(f"\nTop {K} images for concept_{j:04d}:")
for score, idx in zip(v2.tolist(), i2.tolist()):
    print(f"  {score:+.4f}  {all_paths[idx]}")

# --- add after you computed `acts`, `all_paths`, and `idxs` (top concepts) ---
import math, os
from PIL import Image
import matplotlib.pyplot as plt

def plot_top_images_for_concepts(acts, all_paths, concept_indices, k=12, out_dir="concept_grids"):
    os.makedirs(out_dir, exist_ok=True)
    N = acts.shape[0]
    for rank, j in enumerate(concept_indices[:10]):  # top-10 concepts
        col = acts[:, j]  # [N]
        # pick top-k valid paths
        vals, inds = torch.topk(col, k=min(k, N))
        pairs = [(all_paths[i], float(vals[n])) for n, i in enumerate(inds.tolist())]
        pairs = [(p, s) for (p, s) in pairs if p != "<path NA>" and os.path.isfile(p)]
        if not pairs:
            print(f"[warn] concept_{j:04d}: no valid image paths")
            continue
        pairs = pairs[:k]

        # grid size
        cols = 6
        rows = math.ceil(len(pairs) / cols)
        fig = plt.figure(figsize=(cols*2.2, rows*2.2))
        fig.suptitle(f"concept_{j:04d}  (mean act={acts[:, j].mean().item():.4f})", fontsize=12)

        for n, (path, score) in enumerate(pairs):
            ax = fig.add_subplot(rows, cols, n+1)
            try:
                img = Image.open(path).convert("RGB")
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, f"ERR\n{os.path.basename(path)}", ha="center", va="center", fontsize=8)
            ax.set_title(f"{score:.3f}", fontsize=8)
            ax.axis("off")

        fig.tight_layout(rect=[0, 0, 1, 0.96])
        out_path = os.path.join(out_dir, f"concept_{j:04d}_top{k}.png")
        plt.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f"saved: {out_path}")

# Use the same `idxs` you already computed (top by mean activation)
plot_top_images_for_concepts(acts, all_paths, idxs, k=12, out_dir="concept_grids")


Using ImageFolder on pets_structured/. Classes: 35
SAE dims: D=512, F=2048, C=1

Top concepts by mean activation:
  concept_0280: mean_act=0.3235
  concept_0105: mean_act=0.1469
  concept_1207: mean_act=0.1356
  concept_0116: mean_act=0.1281
  concept_1085: mean_act=0.1131
  concept_1366: mean_act=0.1038
  concept_0855: mean_act=0.0921
  concept_0690: mean_act=0.0531
  concept_0406: mean_act=0.0398
  concept_1936: mean_act=0.0397
  concept_0649: mean_act=0.0341
  concept_1317: mean_act=0.0306
  concept_1209: mean_act=0.0287
  concept_0365: mean_act=0.0280
  concept_1894: mean_act=0.0271
  concept_0994: mean_act=0.0248
  concept_1430: mean_act=0.0179
  concept_0918: mean_act=0.0076
  concept_1355: mean_act=0.0069
  concept_1815: mean_act=0.0057

Top 12 images for concept_0280:
  +0.3885  /home/sunayana/Documents/Concept_LoRA/datasets/pets_structured/english/english_setter_42.jpg
  +0.3880  /home/sunayana/Documents/Concept_LoRA/datasets/pets_structured/german/german_shorthaired_158.jpg
 

In [ ]:
import os, math, torch
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import OxfordIIITPet, ImageFolder
import clip
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# ---------------------- paths (edit) ----------------------
PETS_BASE = "/home/sunayana/Documents/Concept_LoRA/datasets"
PETS_OFFICIAL = os.path.join(PETS_BASE, "the-oxfordiiit-pet-dataset")  # must have images/ and annotations/
PETS_STRUCTURED = os.path.join(PETS_BASE, "pets_structured")            # class-per-folder layout
SAE_CKPT = "/home/sunayana/Documents/Concept_LoRA/Discover-then-Name/pretrained/Checkpoints/clip_ViTB16_sparse_autoencoder_final.pt"
OUT_DIR = "concept_eval/pretrained_vitb16_pets"     # output root
TOP_K = 12                        # images per concept
GRID_COLS = 6                     # grid columns (rows auto)

# ---------------------- CLIP ----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()

# ---------------------- SAE ----------------------
from sparse_autoencoder import SparseAutoencoder

def load_sae_from_ckpt(path, device=device):
    obj = torch.load(path, map_location=device, weights_only=True)
    sd = obj.get("state_dict", obj if isinstance(obj, dict) else obj.state_dict())
    if "encoder._weight" in sd and sd["encoder._weight"].ndim == 3:
        C, F, D = sd["encoder._weight"].shape
    elif "decoder._weight" in sd and sd["decoder._weight"].ndim == 3:
        C, D, F = sd["decoder._weight"].shape
    else:
        C, D = sd["tied_bias"].shape
        F = sd["encoder._bias"].shape[-1]
    sae = SparseAutoencoder(n_input_features=int(D), n_learned_features=int(F), n_components=int(C)).to(device)
    sae.load_state_dict(sd, strict=False)
    sae.eval()
    return sae, D, F, C

@torch.no_grad()
def encode_images_to_clip(imgs):
    x = clip_model.encode_image(imgs)
    return x / x.norm(dim=-1, keepdim=True)

# ---------------------- Dataset wrappers ----------------------
class WithPaths(torch.utils.data.Dataset):
    """
    Wrap a dataset to also return the image path with each item.
    Works for both ImageFolder and OxfordIIITPet.
    """
    def __init__(self, base_ds, mode="imagefolder"):
        self.base = base_ds
        self.mode = mode
        if mode == "imagefolder":
            # ImageFolder exposes .samples -> list[(path, class)]
            self.paths = [p for p, _ in self.base.samples]
        elif mode == "oxford":
            # OxfordIIITPet keeps (file, cls) in ._images; files live in 'images/'
            img_root = os.path.join(self.base.root, "images")
            self.paths = [os.path.join(img_root, fname) for fname, _ in self.base._images]
        else:
            raise ValueError("mode must be 'imagefolder' or 'oxford'")

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]  # transforms already applied
        return img, label, self.paths[idx]

def make_loader(batch_size=64, workers=4):
    if os.path.isdir(os.path.join(PETS_OFFICIAL, "images")) and os.path.isdir(os.path.join(PETS_OFFICIAL, "annotations")):
        base = OxfordIIITPet(PETS_OFFICIAL, split="test", target_types="category", download=False, transform=preprocess)
        ds = WithPaths(base, mode="oxford")
        print("Using OxfordIIITPet (official layout).")
    else:
        base = ImageFolder(PETS_STRUCTURED, transform=preprocess)
        ds = WithPaths(base, mode="imagefolder")
        print(f"Using ImageFolder on pets_structured/. Classes: {len(base.classes)}")
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)
    return ds, loader

# ---------------------- Collect activations ----------------------
ds, loader = make_loader(batch_size=64, workers=4)
sae, D, F, C = load_sae_from_ckpt(SAE_CKPT, device)
print(f"SAE dims: D={D}, F={F}, C={C}")

all_acts = []
all_paths = []

with torch.no_grad():
    for imgs, targets, paths in tqdm(loader, desc="Encoding + SAE"):
        imgs = imgs.to(device)
        feats = encode_images_to_clip(imgs)     # [B, D]
        feats = feats.unsqueeze(1)              # [B, 1, D] assuming C=1
        la, _ = sae(feats)                      # [B, 1, F]
        all_acts.append(la.squeeze(1).cpu())    # [B, F]
        all_paths.extend(paths)

acts = torch.cat(all_acts, dim=0)               # [N, F]
N, F = acts.shape
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Collected activations: N={N}, F={F}")
print(f"Saving grids to: {OUT_DIR}")

# ---------------------- Grid saver ----------------------
def save_grid_for_concept(j, scores, k=12, cols=6, out_dir=OUT_DIR):
    pairs = [(all_paths[i], float(scores[i])) for i in torch.topk(scores, k=min(k, scores.numel())).indices.tolist()]
    # keep only valid paths
    pairs = [(p, s) for p, s in pairs if p and p != "<path NA>" and os.path.isfile(p)]
    if not pairs:
        return False

    cols = max(1, cols)
    rows = math.ceil(len(pairs) / cols)
    fig = plt.figure(figsize=(cols * 2.2, rows * 2.2))
    fig.suptitle(f"concept_{j:04d} (mean={scores.mean().item():.4f})", fontsize=12)

    for n, (path, score) in enumerate(pairs[:k]):
        ax = fig.add_subplot(rows, cols, n + 1)
        try:
            img = Image.open(path).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, f"ERR\n{os.path.basename(path)}", ha="center", va="center", fontsize=8)
        ax.set_title(f"{score:.3f}", fontsize=8)
        ax.axis("off")

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    out_path = os.path.join(out_dir, f"concept_{j:04d}_top{k}.png")
    plt.savefig(out_path, dpi=150)
    plt.close(fig)
    return True

# ---------------------- Save grids for ALL neurons ----------------------
mean_acts = acts.mean(dim=0)  # [F] (optional: useful for logging/order)
print("Saving grids for all concepts...")
ok_cnt = 0
for j in tqdm(range(F), desc="Concept grids"):
    ok = save_grid_for_concept(j, acts[:, j], k=TOP_K, cols=GRID_COLS, out_dir=OUT_DIR)
    ok_cnt += int(ok)
print(f"Done. Saved {ok_cnt}/{F} grids to {OUT_DIR}.")


Using ImageFolder on pets_structured/. Classes: 35
SAE dims: D=512, F=4096, C=1


Encoding + SAE: 100%|██████████| 116/116 [00:10<00:00, 11.46it/s]


Collected activations: N=7390, F=4096
Saving grids to: concept_eval/pretrained_vitb16_pets
Saving grids for all concepts...


Concept grids: 100%|██████████| 4096/4096 [1:07:10<00:00,  1.02it/s]

Done. Saved 4096/4096 grids to concept_eval/pretrained_vitb16_pets.


: 

In [ ]:
import os, torch
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import OxfordIIITPet, ImageFolder
import clip

# --- paths (edit to your absolute paths) ---
CUB_BASE = "/home/sunayana/Documents/Concept_LoRA/datasets"
CUB_STRUCTURED = os.path.join(CUB_BASE, "cub2002011/images")            # your class-per-folder layout
SAE_CKPT = "/home/sunayana/Documents/Concept_LoRA/Discover-then-Name/pretrained/Checkpoints/clip_ViTB16_sparse_autoencoder_final.pt"

# --- CLIP encoder ---
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()

# --- dataset loader chooser ---
def make_loader(batch_size=64, workers=4):

    # Fallback: your structured layout (class folders)
    ds = ImageFolder(CUB_STRUCTURED, transform=preprocess)
    print("Using ImageFolder on cub2002011/images/ Classes:", len(ds.classes))
    
    return ds, DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)

# --- load SAE (keeps component axis) ---
from sparse_autoencoder import SparseAutoencoder
def load_sae_from_ckpt(path, device=device):
    obj = torch.load(path, map_location=device, weights_only=True)
    sd = obj["state_dict"] if isinstance(obj, dict) and "state_dict" in obj else (obj if isinstance(obj, dict) else obj.state_dict())
    if "encoder._weight" in sd and sd["encoder._weight"].ndim == 3:
        C, F, D = sd["encoder._weight"].shape
    elif "decoder._weight" in sd and sd["decoder._weight"].ndim == 3:
        C, D, F = sd["decoder._weight"].shape
    else:
        C, D = sd["tied_bias"].shape; F = sd["encoder._bias"].shape[-1]
    sae = SparseAutoencoder(n_input_features=int(D), n_learned_features=int(F), n_components=int(C)).to(device)
    sae.load_state_dict(sd, strict=False); sae.eval()
    return sae, D, F, C

@torch.no_grad()
def encode_images_to_clip(imgs):
    x = clip_model.encode_image(imgs)
    x = x / x.norm(dim=-1, keepdim=True)
    return x

#load pets dataset from the root using ImageFolder
from torch.utils.data import ConcatDataset
# Fallback: your structured layout (class folders)
def build_dataset(root, preprocess):
    p = os.path.join(root, "cub2002011", "images")
    if os.path.isdir(p): return ImageFolder(p, transform=preprocess), p
    p = os.path.join(root, "images")
    if os.path.isdir(p): return ImageFolder(p, transform=preprocess), p
    base = os.path.join(root, "cub2002011")
    if all(os.path.isdir(os.path.join(base, s)) for s in ("train", "val", "test")):
        parts = [ImageFolder(os.path.join(base, s), transform=preprocess) for s in ("train", "val", "test")]
        return ConcatDataset(parts), base
    if all(os.path.isdir(os.path.join(root, s)) for s in ("train", "val", "test")):
        parts = [ImageFolder(os.path.join(root, s), transform=preprocess) for s in ("train", "val", "test")]
        return ConcatDataset(parts), root
    raise FileNotFoundError(f"Could not find CUB images under {root}.")

def collect_paths(ds):
    if hasattr(ds, "samples"):
        return [p for p, _ in ds.samples]
    if isinstance(ds, ConcatDataset):
        paths = []
        for sub in ds.datasets:
            if not hasattr(sub, "samples"):
                raise TypeError("ConcatDataset contains non-ImageFolder subset.")
            paths.extend([p for p, _ in sub.samples])
        return paths
    raise TypeError(f"Unsupported dataset type: {type(ds)}")

ds, root = build_dataset(CUB_BASE, preprocess)


loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)



sae, D, F, C = load_sae_from_ckpt(SAE_CKPT, device)
print(f"SAE dims: D={D}, F={F}, C={C}")

all_acts = []
all_paths = []

with torch.no_grad():
    for imgs, targets in loader:
        imgs = imgs.to(device)
        feats = encode_images_to_clip(imgs)       # [B, D]
        feats = feats.unsqueeze(1)                # [B, 1, D] for C=1
        la, _ = sae(feats)                        # learned activations [B, 1, F]
        all_acts.append(la.squeeze(1).cpu())      # [B, F]
        # robust path extraction
        if hasattr(ds, "imgs"):
            all_paths += [p for p,_ in ds.imgs[len(all_paths):len(all_paths)+imgs.size(0)]]
        elif hasattr(ds, "_images"):
            all_paths += [os.path.join(PETS_OFFICIAL, "images", fn) for fn,_ in ds._images[len(all_paths):len(all_paths)+imgs.size(0)]]
        else:
            all_paths += ["<path NA>"] * imgs.size(0)

acts = torch.cat(all_acts, dim=0)                 # [N, F]

# ----- top concepts (dataset-wide mean activation) -----
topk = 20
vals, idxs = torch.topk(acts.mean(dim=0), k=topk)
print("\nTop concepts by mean activation:")
for v, i in zip(vals.tolist(), idxs.tolist()):
    print(f"  concept_{i:04d}: mean_act={v:.4f}")

# ----- top images for the strongest concept -----
K = 12
j = idxs[0].item()
col = acts[:, j]
v2, i2 = torch.topk(col, k=min(K, col.numel()))
print(f"\nTop {K} images for concept_{j:04d}:")
for score, idx in zip(v2.tolist(), i2.tolist()):
    print(f"  {score:+.4f}  {all_paths[idx]}")

# --- add after you computed `acts`, `all_paths`, and `idxs` (top concepts) ---
import math, os
from PIL import Image
import matplotlib.pyplot as plt

def plot_top_images_for_concepts(acts, all_paths, concept_indices, k=12, out_dir="concept_grids"):
    os.makedirs(out_dir, exist_ok=True)
    N = acts.shape[0]
    for rank, j in enumerate(concept_indices[:10]):  # top-10 concepts
        col = acts[:, j]  # [N]
        # pick top-k valid paths
        vals, inds = torch.topk(col, k=min(k, N))
        pairs = [(all_paths[i], float(vals[n])) for n, i in enumerate(inds.tolist())]
        pairs = [(p, s) for (p, s) in pairs if p != "<path NA>" and os.path.isfile(p)]
        if not pairs:
            print(f"[warn] concept_{j:04d}: no valid image paths")
            continue
        pairs = pairs[:k]

        # grid size
        cols = 6
        rows = math.ceil(len(pairs) / cols)
        fig = plt.figure(figsize=(cols*2.2, rows*2.2))
        fig.suptitle(f"concept_{j:04d}  (mean act={acts[:, j].mean().item():.4f})", fontsize=12)

        for n, (path, score) in enumerate(pairs):
            ax = fig.add_subplot(rows, cols, n+1)
            try:
                img = Image.open(path).convert("RGB")
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, f"ERR\n{os.path.basename(path)}", ha="center", va="center", fontsize=8)
            ax.set_title(f"{score:.3f}", fontsize=8)
            ax.axis("off")

        fig.tight_layout(rect=[0, 0, 1, 0.96])
        out_path = os.path.join(out_dir, f"concept_{j:04d}_top{k}.png")
        plt.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f"saved: {out_path}")

# Use the same `idxs` you already computed (top by mean activation)
plot_top_images_for_concepts(acts, all_paths, idxs, k=12, out_dir="concept_grids_cub_lora")


In [5]:
import os
import math
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, ConcatDataset
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
import clip

# --- paths (edit to your absolute paths) ---
CUB_BASE = "/home/sunayana/Documents/Concept_LoRA/datasets"
SAE_CKPT = "SAE/SAEImg/cc3m/clip_RN50/out/lr0.0001_l1coeff0.0003_ef4_rf500000_hookout_bs4096_epo200/sae_checkpoints/sparse_autoencoder_final.pt"

# --- CLIP encoder ---
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()

# --- Dataset loading ---
def build_dataset(root, preprocess):
    base = os.path.join(root, "cub2002011")

    # Load from train/val/test
    if all(os.path.isdir(os.path.join(base, s)) for s in ("train", "val", "test")):
        print("Using CUB dataset from train/val/test splits")
        parts = [ImageFolder(os.path.join(base, s), transform=preprocess) for s in ("train", "val", "test")]
        return ConcatDataset(parts), base

    # fallback: check root directly
    if all(os.path.isdir(os.path.join(root, s)) for s in ("train", "val", "test")):
        print("Using dataset from train/val/test splits in root")
        parts = [ImageFolder(os.path.join(root, s), transform=preprocess) for s in ("train", "val", "test")]
        return ConcatDataset(parts), root

    raise FileNotFoundError(f"Could not find expected CUB train/val/test folders under {root}")

def collect_paths(ds):
    if hasattr(ds, "samples"):
        return [p for p, _ in ds.samples]
    if isinstance(ds, ConcatDataset):
        paths = []
        for sub in ds.datasets:
            if not hasattr(sub, "samples"):
                raise TypeError("ConcatDataset contains non-ImageFolder subset.")
            paths.extend([p for p, _ in sub.samples])
        return paths
    raise TypeError(f"Unsupported dataset type: {type(ds)}")

# --- CLIP feature extraction ---
@torch.no_grad()
def encode_images_to_clip(imgs):
    x = clip_model.encode_image(imgs)
    x = x / x.norm(dim=-1, keepdim=True)
    return x

# --- Load SAE ---
from sparse_autoencoder import SparseAutoencoder

def load_sae_from_ckpt(path, device=device):
    obj = torch.load(path, map_location=device, weights_only=True)
    sd = obj["state_dict"] if isinstance(obj, dict) and "state_dict" in obj else (obj if isinstance(obj, dict) else obj.state_dict())
    if "encoder._weight" in sd and sd["encoder._weight"].ndim == 3:
        C, F, D = sd["encoder._weight"].shape
    elif "decoder._weight" in sd and sd["decoder._weight"].ndim == 3:
        C, D, F = sd["decoder._weight"].shape
    else:
        C, D = sd["tied_bias"].shape
        F = sd["encoder._bias"].shape[-1]
    sae = SparseAutoencoder(n_input_features=int(D), n_learned_features=int(F), n_components=int(C)).to(device)
    sae.load_state_dict(sd, strict=False)
    sae.eval()
    return sae, D, F, C

# --- Plot top-K images per concept ---
def plot_top_images_for_concepts(acts, all_paths, concept_indices, k=12, out_dir="concept_grids"):
    os.makedirs(out_dir, exist_ok=True)
    N = acts.shape[0]
    for rank, j in enumerate(concept_indices[:10]):  # top-10 concepts
        col = acts[:, j]  # [N]
        vals, inds = torch.topk(col, k=min(k, N))
        pairs = [(all_paths[i], float(vals[n])) for n, i in enumerate(inds.tolist())]
        pairs = [(p, s) for (p, s) in pairs if p != "<path NA>" and os.path.isfile(p)]
        if not pairs:
            print(f"[warn] concept_{j:04d}: no valid image paths")
            continue
        pairs = pairs[:k]

        # Grid
        cols = 6
        rows = math.ceil(len(pairs) / cols)
        fig = plt.figure(figsize=(cols*2.2, rows*2.2))
        fig.suptitle(f"concept_{j:04d}  (mean act={acts[:, j].mean().item():.4f})", fontsize=12)

        for n, (path, score) in enumerate(pairs):
            ax = fig.add_subplot(rows, cols, n+1)
            try:
                img = Image.open(path).convert("RGB")
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, f"ERR\n{os.path.basename(path)}", ha="center", va="center", fontsize=8)
            ax.set_title(f"{score:.3f}", fontsize=8)
            ax.axis("off")

        fig.tight_layout(rect=[0, 0, 1, 0.96])
        out_path = os.path.join(out_dir, f"concept_{j:04d}_top{k}.png")
        plt.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f"saved: {out_path}")

# -------------------
# Pipeline starts here
# -------------------

# Load dataset
ds, root = build_dataset(CUB_BASE, preprocess)
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
all_paths = collect_paths(ds)

# Load SAE
sae, D, F, C = load_sae_from_ckpt(SAE_CKPT, device)
print(f"SAE dims: D={D}, F={F}, C={C}")

# Compute activations
all_acts = []

with torch.no_grad():
    for imgs, _ in loader:
        imgs = imgs.to(device)
        feats = encode_images_to_clip(imgs)       # [B, D]
        feats = feats.unsqueeze(1)                # [B, 1, D] for C=1
        la, _ = sae(feats)                        # [B, 1, F]
        all_acts.append(la.squeeze(1).cpu())      # [B, F]

acts = torch.cat(all_acts, dim=0)                 # [N, F]

# Top concepts
topk = 20
vals, idxs = torch.topk(acts.mean(dim=0), k=topk)
print("\nTop concepts by mean activation:")
for v, i in zip(vals.tolist(), idxs.tolist()):
    print(f"  concept_{i:04d}: mean_act={v:.4f}")

# Top images for top concept
K = 12
j = idxs[0].item()
col = acts[:, j]
v2, i2 = torch.topk(col, k=min(K, col.numel()))
print(f"\nTop {K} images for concept_{j:04d}:")
for score, idx in zip(v2.tolist(), i2.tolist()):
    print(f"  {score:+.4f}  {all_paths[idx]}")

# Plot top images for top concepts
plot_top_images_for_concepts(acts, all_paths, idxs, k=12, out_dir="concept_grids_cub_lora")


Using CUB dataset from train/val/test splits
SAE dims: D=512, F=2048, C=1

Top concepts by mean activation:
  concept_0427: mean_act=0.1103
  concept_1430: mean_act=0.0753
  concept_0280: mean_act=0.0625
  concept_0502: mean_act=0.0328
  concept_1085: mean_act=0.0251
  concept_0077: mean_act=0.0248
  concept_0737: mean_act=0.0225
  concept_0009: mean_act=0.0212
  concept_1417: mean_act=0.0211
  concept_0771: mean_act=0.0183
  concept_0116: mean_act=0.0180
  concept_1195: mean_act=0.0172
  concept_0830: mean_act=0.0168
  concept_1364: mean_act=0.0166
  concept_1958: mean_act=0.0151
  concept_1853: mean_act=0.0145
  concept_0874: mean_act=0.0129
  concept_0918: mean_act=0.0128
  concept_0666: mean_act=0.0118
  concept_1335: mean_act=0.0112

Top 12 images for concept_0427:
  +0.4141  /home/sunayana/Documents/Concept_LoRA/datasets/cub2002011/train/001.Black_footed_Albatross/Black_Footed_Albatross_0077_796114.jpg
  +0.4070  /home/sunayana/Documents/Concept_LoRA/datasets/cub2002011/train/002

In [6]:
import os
import math
import csv
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, ConcatDataset
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
import clip

# --- paths ---
CUB_BASE = "/home/sunayana/Documents/Concept_LoRA/datasets"
SAE_CKPT = "SAE/SAEImg/cc3m/clip_RN50/out/lr0.0001_l1coeff0.0003_ef4_rf500000_hookout_bs4096_epo200/sae_checkpoints/sparse_autoencoder_final.pt"

# --- CLIP encoder ---
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()

# --- Dataset loading ---
def build_dataset(root, preprocess):
    base = os.path.join(root, "cub2002011")
    if all(os.path.isdir(os.path.join(base, s)) for s in ("train", "val", "test")):
        print("Using CUB dataset from train/val/test splits")
        parts = [ImageFolder(os.path.join(base, s), transform=preprocess) for s in ("train", "val", "test")]
        return ConcatDataset(parts), base
    if all(os.path.isdir(os.path.join(root, s)) for s in ("train", "val", "test")):
        print("Using dataset from train/val/test splits in root")
        parts = [ImageFolder(os.path.join(root, s), transform=preprocess) for s in ("train", "val", "test")]
        return ConcatDataset(parts), root
    raise FileNotFoundError(f"Could not find expected train/val/test folders under {root}")

def collect_paths_and_labels(ds):
    if hasattr(ds, "samples"):
        return [p for p, _ in ds.samples], [int(t) for _, t in ds.samples]
    if isinstance(ds, ConcatDataset):
        paths, labels = [], []
        for sub in ds.datasets:
            if not hasattr(sub, "samples"):
                raise TypeError("ConcatDataset contains non-ImageFolder subset.")
            paths.extend([p for p, _ in sub.samples])
            labels.extend([int(t) for _, t in sub.samples])
        return paths, labels
    raise TypeError(f"Unsupported dataset type: {type(ds)}")

# --- CLIP feature extraction ---
@torch.no_grad()
def encode_images_to_clip(imgs):
    x = clip_model.encode_image(imgs)
    x = x / x.norm(dim=-1, keepdim=True)
    return x

# --- Load SAE ---
from sparse_autoencoder import SparseAutoencoder

def load_sae_from_ckpt(path, device=device):
    obj = torch.load(path, map_location=device, weights_only=True)
    sd = obj["state_dict"] if isinstance(obj, dict) and "state_dict" in obj else (obj if isinstance(obj, dict) else obj.state_dict())
    if "encoder._weight" in sd and sd["encoder._weight"].ndim == 3:
        C, F, D = sd["encoder._weight"].shape
    elif "decoder._weight" in sd and sd["decoder._weight"].ndim == 3:
        C, D, F = sd["decoder._weight"].shape
    else:
        C, D = sd["tied_bias"].shape
        F = sd["encoder._bias"].shape[-1]
    sae = SparseAutoencoder(n_input_features=int(D), n_learned_features=int(F), n_components=int(C)).to(device)
    sae.load_state_dict(sd, strict=False)
    sae.eval()
    return sae, D, F, C

# --- Plot + Save top-K images + Metadata ---
def plot_top_images_and_save_metadata(
    acts, all_paths, all_labels, class_names, out_dir="concept_grids_all", k=12
):
    os.makedirs(out_dir, exist_ok=True)
    N, F = acts.shape

    metadata_file = os.path.join(out_dir, "concept_metadata.csv")
    with open(metadata_file, mode="w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["concept_id", "image_path", "score", "class_id", "class_name"])

        for j in range(F):
            col = acts[:, j]  # [N]
            vals, inds = torch.topk(col, k=min(k, N))
            pairs = [(all_paths[i], float(vals[n]), all_labels[i]) for n, i in enumerate(inds.tolist())]
            pairs = [(p, s, c) for (p, s, c) in pairs if os.path.isfile(p)]
            if not pairs:
                print(f"[warn] concept_{j:04d}: no valid image paths")
                continue
            pairs = pairs[:k]

            # Save metadata
            for (path, score, class_id) in pairs:
                writer.writerow([j, path, score, class_id, class_names[class_id]])

            # Save image grid
            cols = 6
            rows = math.ceil(len(pairs) / cols)
            fig = plt.figure(figsize=(cols*2.2, rows*2.2))
            fig.suptitle(f"concept_{j:04d}  (mean act={acts[:, j].mean().item():.4f})", fontsize=12)

            for n, (path, score, _) in enumerate(pairs):
                ax = fig.add_subplot(rows, cols, n+1)
                try:
                    img = Image.open(path).convert("RGB")
                    ax.imshow(img)
                except Exception:
                    ax.text(0.5, 0.5, "ERR", ha="center", va="center", fontsize=8)
                ax.set_title(f"{score:.3f}", fontsize=8)
                ax.axis("off")

            fig.tight_layout(rect=[0, 0, 1, 0.96])
            out_path = os.path.join(out_dir, f"concept_{j:04d}_top{k}.png")
            plt.savefig(out_path, dpi=150)
            plt.close(fig)
            print(f"[✓] concept_{j:04d}: saved to {out_path}")

# -------------------
# Pipeline starts here
# -------------------

# Load dataset
ds, root = build_dataset(CUB_BASE, preprocess)
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
all_paths, all_labels = collect_paths_and_labels(ds)
class_names = ds.datasets[0].classes if isinstance(ds, ConcatDataset) else ds.classes

# Load SAE
sae, D, F, C = load_sae_from_ckpt(SAE_CKPT, device)
print(f"SAE dims: D={D}, F={F}, C={C}")

# Compute activations
all_acts = []

with torch.no_grad():
    for imgs, _ in loader:
        imgs = imgs.to(device)
        feats = encode_images_to_clip(imgs)       # [B, D]
        feats = feats.unsqueeze(1)                # [B, 1, D] for C=1
        la, _ = sae(feats)                        # [B, 1, F]
        all_acts.append(la.squeeze(1).cpu())      # [B, F]

acts = torch.cat(all_acts, dim=0)                 # [N, F]

# Top concepts summary
topk = 20
vals, idxs = torch.topk(acts.mean(dim=0), k=topk)
print("\nTop concepts by mean activation:")
for v, i in zip(vals.tolist(), idxs.tolist()):
    print(f"  concept_{i:04d}: mean_act={v:.4f}")

# Save image grids + metadata
plot_top_images_and_save_metadata(
    acts, all_paths, all_labels, class_names,
    out_dir="concept_grids_all", k=12
)


Using CUB dataset from train/val/test splits
SAE dims: D=512, F=2048, C=1

Top concepts by mean activation:
  concept_0427: mean_act=0.1103
  concept_1430: mean_act=0.0753
  concept_0280: mean_act=0.0625
  concept_0502: mean_act=0.0328
  concept_1085: mean_act=0.0251
  concept_0077: mean_act=0.0248
  concept_0737: mean_act=0.0225
  concept_0009: mean_act=0.0212
  concept_1417: mean_act=0.0211
  concept_0771: mean_act=0.0183
  concept_0116: mean_act=0.0180
  concept_1195: mean_act=0.0172
  concept_0830: mean_act=0.0168
  concept_1364: mean_act=0.0166
  concept_1958: mean_act=0.0151
  concept_1853: mean_act=0.0145
  concept_0874: mean_act=0.0129
  concept_0918: mean_act=0.0128
  concept_0666: mean_act=0.0118
  concept_1335: mean_act=0.0112
[✓] concept_0000: saved to concept_grids_all/concept_0000_top12.png
[✓] concept_0001: saved to concept_grids_all/concept_0001_top12.png
[✓] concept_0002: saved to concept_grids_all/concept_0002_top12.png
[✓] concept_0003: saved to concept_grids_all/con

In [7]:



import os
import math
import csv
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, ConcatDataset
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
import clip

# --- paths ---
CUB_BASE = "/home/sunayana/Documents/Concept_LoRA/datasets"
SAE_CKPT = "/home/sunayana/Documents/Concept_LoRA/Discover-then-Name/SAE/SAEImg/cub_regular/clip_ViT-B16/out/lr0.0001_l1coeff0.0003_ef4_rf500000_hookout_bs4096_epo200/sae_checkpoints_cub_vitb16_d512/sparse_autoencoder_final.pt"

# --- CLIP encoder ---
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()

# --- Dataset loading ---
def build_dataset(root, preprocess):
    base = os.path.join(root, "cub2002011")
    if all(os.path.isdir(os.path.join(base, s)) for s in ("train", "val", "test")):
        print("Using CUB dataset from train/val/test splits")
        parts = [ImageFolder(os.path.join(base, s), transform=preprocess) for s in ("train", "val", "test")]
        return ConcatDataset(parts), base
    if all(os.path.isdir(os.path.join(root, s)) for s in ("train", "val", "test")):
        print("Using dataset from train/val/test splits in root")
        parts = [ImageFolder(os.path.join(root, s), transform=preprocess) for s in ("train", "val", "test")]
        return ConcatDataset(parts), root
    raise FileNotFoundError(f"Could not find expected train/val/test folders under {root}")

def collect_paths_and_labels(ds):
    if hasattr(ds, "samples"):
        return [p for p, _ in ds.samples], [int(t) for _, t in ds.samples]
    if isinstance(ds, ConcatDataset):
        paths, labels = [], []
        for sub in ds.datasets:
            if not hasattr(sub, "samples"):
                raise TypeError("ConcatDataset contains non-ImageFolder subset.")
            paths.extend([p for p, _ in sub.samples])
            labels.extend([int(t) for _, t in sub.samples])
        return paths, labels
    raise TypeError(f"Unsupported dataset type: {type(ds)}")

# --- CLIP feature extraction ---
@torch.no_grad()
def encode_images_to_clip(imgs):
    x = clip_model.encode_image(imgs)
    x = x / x.norm(dim=-1, keepdim=True)
    return x

# --- Load SAE ---
from sparse_autoencoder import SparseAutoencoder

def load_sae_from_ckpt(path, device=device):
    obj = torch.load(path, map_location=device, weights_only=True)
    sd = obj["state_dict"] if isinstance(obj, dict) and "state_dict" in obj else (obj if isinstance(obj, dict) else obj.state_dict())
    if "encoder._weight" in sd and sd["encoder._weight"].ndim == 3:
        C, F, D = sd["encoder._weight"].shape
    elif "decoder._weight" in sd and sd["decoder._weight"].ndim == 3:
        C, D, F = sd["decoder._weight"].shape
    else:
        C, D = sd["tied_bias"].shape
        F = sd["encoder._bias"].shape[-1]
    sae = SparseAutoencoder(n_input_features=int(D), n_learned_features=int(F), n_components=int(C)).to(device)
    sae.load_state_dict(sd, strict=False)
    sae.eval()
    return sae, D, F, C

# --- Plot + Save top-K images + Metadata ---
def plot_top_images_and_save_metadata(
    acts, all_paths, all_labels, class_names, out_dir="concept_grids_all", k=12
):
    os.makedirs(out_dir, exist_ok=True)
    N, F = acts.shape

    metadata_file = os.path.join(out_dir, "concept_metadata.csv")
    with open(metadata_file, mode="w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["concept_id", "image_path", "score", "class_id", "class_name"])

        for j in range(F):
            col = acts[:, j]  # [N]
            vals, inds = torch.topk(col, k=min(k, N))
            pairs = [(all_paths[i], float(vals[n]), all_labels[i]) for n, i in enumerate(inds.tolist())]
            pairs = [(p, s, c) for (p, s, c) in pairs if os.path.isfile(p)]
            if not pairs:
                print(f"[warn] concept_{j:04d}: no valid image paths")
                continue
            pairs = pairs[:k]

            # Save metadata
            for (path, score, class_id) in pairs:
                writer.writerow([j, path, score, class_id, class_names[class_id]])

            # Save image grid
            cols = 6
            rows = math.ceil(len(pairs) / cols)
            fig = plt.figure(figsize=(cols*2.2, rows*2.2))
            fig.suptitle(f"concept_{j:04d}  (mean act={acts[:, j].mean().item():.4f})", fontsize=12)

            for n, (path, score, _) in enumerate(pairs):
                ax = fig.add_subplot(rows, cols, n+1)
                try:
                    img = Image.open(path).convert("RGB")
                    ax.imshow(img)
                except Exception:
                    ax.text(0.5, 0.5, "ERR", ha="center", va="center", fontsize=8)
                ax.set_title(f"{score:.3f}", fontsize=8)
                ax.axis("off")

            fig.tight_layout(rect=[0, 0, 1, 0.96])
            out_path = os.path.join(out_dir, f"concept_{j:04d}_top{k}.png")
            plt.savefig(out_path, dpi=150)
            plt.close(fig)
            print(f"[✓] concept_{j:04d}: saved to {out_path}")

# -------------------
# Pipeline starts here
# -------------------

# Load dataset
ds, root = build_dataset(CUB_BASE, preprocess)
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
all_paths, all_labels = collect_paths_and_labels(ds)
class_names = ds.datasets[0].classes if isinstance(ds, ConcatDataset) else ds.classes

# Load SAE
sae, D, F, C = load_sae_from_ckpt(SAE_CKPT, device)
print(f"SAE dims: D={D}, F={F}, C={C}")

# Compute activations
all_acts = []

with torch.no_grad():
    for imgs, _ in loader:
        imgs = imgs.to(device)
        feats = encode_images_to_clip(imgs)       # [B, D]
        feats = feats.unsqueeze(1)                # [B, 1, D] for C=1
        la, _ = sae(feats)                        # [B, 1, F]
        all_acts.append(la.squeeze(1).cpu())      # [B, F]

acts = torch.cat(all_acts, dim=0)                 # [N, F]

# Top concepts summary
topk = 20
vals, idxs = torch.topk(acts.mean(dim=0), k=topk)
print("\nTop concepts by mean activation:")
for v, i in zip(vals.tolist(), idxs.tolist()):
    print(f"  concept_{i:04d}: mean_act={v:.4f}")

# Save image grids + metadata
plot_top_images_and_save_metadata(
    acts, all_paths, all_labels, class_names,
    out_dir="concept_grids_all_base", k=12
)



Using CUB dataset from train/val/test splits
SAE dims: D=512, F=2048, C=1

Top concepts by mean activation:
  concept_0280: mean_act=0.3288
  concept_1417: mean_act=0.1825
  concept_0830: mean_act=0.0954
  concept_0116: mean_act=0.0865
  concept_0537: mean_act=0.0653
  concept_1430: mean_act=0.0476
  concept_0009: mean_act=0.0433
  concept_1736: mean_act=0.0358
  concept_1766: mean_act=0.0335
  concept_0376: mean_act=0.0223
  concept_0105: mean_act=0.0221
  concept_0237: mean_act=0.0196
  concept_1592: mean_act=0.0177
  concept_0855: mean_act=0.0124
  concept_0502: mean_act=0.0086
  concept_0771: mean_act=0.0081
  concept_1355: mean_act=0.0059
  concept_1857: mean_act=0.0050
  concept_0427: mean_act=0.0049
  concept_0994: mean_act=0.0043
[✓] concept_0000: saved to concept_grids_all_base/concept_0000_top12.png
[✓] concept_0001: saved to concept_grids_all_base/concept_0001_top12.png
[✓] concept_0002: saved to concept_grids_all_base/concept_0002_top12.png
[✓] concept_0003: saved to concep

In [ ]:
import os, torch
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import OxfordIIITPet, ImageFolder
import clip

# --- paths (edit to your absolute paths) ---
PETS_BASE = "/home/sunayana/Documents/Concept_LoRA/datasets"
PETS_OFFICIAL = os.path.join(PETS_BASE, "the-oxfordiiit-pet-dataset")  # must contain images/ and annotations/
PETS_STRUCTURED = os.path.join(PETS_BASE, "pets_structured")            # your class-per-folder layout
SAE_CKPT = "/home/sunayana/Documents/Concept_LoRA/Discover-then-Name/SAE/SAEImg/pets/clip_ViT-B16/out/lr1e-05_l1coeff0.0003_ef8_rf500000_hookout_bs4096_epo50/sae_checkpoints_pets_vitb16_d512_finetuned/sparse_autoencoder_final.pt"

# --- CLIP encoder ---
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()

# --- dataset loader chooser ---
def make_loader(batch_size=64, workers=4):
    if os.path.isdir(os.path.join(PETS_OFFICIAL, "images")) and os.path.isdir(os.path.join(PETS_OFFICIAL, "annotations")):
        ds = OxfordIIITPet(PETS_OFFICIAL, split="test", target_types="category", download=False, transform=preprocess)
        print("Using OxfordIIITPet (official layout).")
    else:
        # Fallback: your structured layout (class folders)
        ds = ImageFolder(PETS_STRUCTURED, transform=preprocess)
        print("Using ImageFolder on pets_structured/. Classes:", len(ds.classes))
    return ds, DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)

# --- load SAE (keeps component axis) ---
from sparse_autoencoder import SparseAutoencoder
def load_sae_from_ckpt(path, device=device):
    obj = torch.load(path, map_location=device, weights_only=True)
    sd = obj["state_dict"] if isinstance(obj, dict) and "state_dict" in obj else (obj if isinstance(obj, dict) else obj.state_dict())
    if "encoder._weight" in sd and sd["encoder._weight"].ndim == 3:
        C, F, D = sd["encoder._weight"].shape
    elif "decoder._weight" in sd and sd["decoder._weight"].ndim == 3:
        C, D, F = sd["decoder._weight"].shape
    else:
        C, D = sd["tied_bias"].shape; F = sd["encoder._bias"].shape[-1]
    sae = SparseAutoencoder(n_input_features=int(D), n_learned_features=int(F), n_components=int(C)).to(device)
    sae.load_state_dict(sd, strict=False); sae.eval()
    return sae, D, F, C

@torch.no_grad()
def encode_images_to_clip(imgs):
    x = clip_model.encode_image(imgs)
    x = x / x.norm(dim=-1, keepdim=True)
    return x

#load pets dataset from the root using ImageFolder
if os.path.isdir(os.path.join(PETS_OFFICIAL, "images")) and os.path.isdir(os.path.join(PETS_OFFICIAL, "annotations")):
    ds = OxfordIIITPet(PETS_OFFICIAL, split="test", target_types="category", download=False, transform=preprocess)
    print("Using OxfordIIITPet (official layout).")
else:
    # Fallback: your structured layout (class folders)
    ds = ImageFolder(PETS_STRUCTURED, transform=preprocess)
    print("Using ImageFolder on pets_structured/. Classes:", len(ds.classes)) 
      
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)



sae, D, F, C = load_sae_from_ckpt(SAE_CKPT, device)
print(f"SAE dims: D={D}, F={F}, C={C}")

all_acts = []
all_paths = []

with torch.no_grad():
    for imgs, targets in loader:
        imgs = imgs.to(device)
        feats = encode_images_to_clip(imgs)       # [B, D]
        feats = feats.unsqueeze(1)                # [B, 1, D] for C=1
        la, _ = sae(feats)                        # learned activations [B, 1, F]
        all_acts.append(la.squeeze(1).cpu())      # [B, F]
        # robust path extraction
        if hasattr(ds, "imgs"):
            all_paths += [p for p,_ in ds.imgs[len(all_paths):len(all_paths)+imgs.size(0)]]
        elif hasattr(ds, "_images"):
            all_paths += [os.path.join(PETS_OFFICIAL, "images", fn) for fn,_ in ds._images[len(all_paths):len(all_paths)+imgs.size(0)]]
        else:
            all_paths += ["<path NA>"] * imgs.size(0)

acts = torch.cat(all_acts, dim=0)                 # [N, F]

# ----- top concepts (dataset-wide mean activation) -----
topk = 20
vals, idxs = torch.topk(acts.mean(dim=0), k=topk)
print("\nTop concepts by mean activation:")
for v, i in zip(vals.tolist(), idxs.tolist()):
    print(f"  concept_{i:04d}: mean_act={v:.4f}")

# ----- top images for the strongest concept -----
K = 12
j = idxs[0].item()
col = acts[:, j]
v2, i2 = torch.topk(col, k=min(K, col.numel()))
print(f"\nTop {K} images for concept_{j:04d}:")
for score, idx in zip(v2.tolist(), i2.tolist()):
    print(f"  {score:+.4f}  {all_paths[idx]}")

# --- add after you computed `acts`, `all_paths`, and `idxs` (top concepts) ---
import math, os
from PIL import Image
import matplotlib.pyplot as plt

def plot_top_images_for_concepts(acts, all_paths, concept_indices, k=12, out_dir="concept_grids"):
    os.makedirs(out_dir, exist_ok=True)
    N = acts.shape[0]
    for rank, j in enumerate(concept_indices[:10]):  # top-10 concepts
        col = acts[:, j]  # [N]
        # pick top-k valid paths
        vals, inds = torch.topk(col, k=min(k, N))
        pairs = [(all_paths[i], float(vals[n])) for n, i in enumerate(inds.tolist())]
        pairs = [(p, s) for (p, s) in pairs if p != "<path NA>" and os.path.isfile(p)]
        if not pairs:
            print(f"[warn] concept_{j:04d}: no valid image paths")
            continue
        pairs = pairs[:k]

        # grid size
        cols = 6
        rows = math.ceil(len(pairs) / cols)
        fig = plt.figure(figsize=(cols*2.2, rows*2.2))
        fig.suptitle(f"concept_{j:04d}  (mean act={acts[:, j].mean().item():.4f})", fontsize=12)

        for n, (path, score) in enumerate(pairs):
            ax = fig.add_subplot(rows, cols, n+1)
            try:
                img = Image.open(path).convert("RGB")
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, f"ERR\n{os.path.basename(path)}", ha="center", va="center", fontsize=8)
            ax.set_title(f"{score:.3f}", fontsize=8)
            ax.axis("off")

        fig.tight_layout(rect=[0, 0, 1, 0.96])
        out_path = os.path.join(out_dir, f"concept_{j:04d}_top{k}.png")
        plt.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f"saved: {out_path}")

# Use the same `idxs` you already computed (top by mean activation)
plot_top_images_for_concepts(acts, all_paths, idxs, k=12, out_dir="concept_grids_pets_finetuned")



Using ImageFolder on pets_structured/. Classes: 35
SAE dims: D=512, F=4096, C=1

Top concepts by mean activation:
  concept_3499: mean_act=2.4665
  concept_2042: mean_act=1.1258
  concept_0462: mean_act=0.7895
  concept_0183: mean_act=0.7665
  concept_3539: mean_act=0.6493
  concept_2473: mean_act=0.6362
  concept_2956: mean_act=0.5445
  concept_2281: mean_act=0.5330
  concept_0671: mean_act=0.5179
  concept_0229: mean_act=0.4889
  concept_1034: mean_act=0.4705
  concept_0514: mean_act=0.4665
  concept_1254: mean_act=0.4592
  concept_3239: mean_act=0.4530
  concept_2192: mean_act=0.4509
  concept_1399: mean_act=0.4506
  concept_2330: mean_act=0.4497
  concept_0840: mean_act=0.4358
  concept_0999: mean_act=0.4302
  concept_1279: mean_act=0.4192

Top 12 images for concept_3499:
  +2.5247  /home/sunayana/Documents/Concept_LoRA/datasets/pets_structured/samoyed/samoyed_138.jpg
  +2.5233  /home/sunayana/Documents/Concept_LoRA/datasets/pets_structured/shiba/shiba_inu_102.jpg
  +2.5217  /home/